# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [10]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [11]:
# TODO: Import the necessary libs
# For example: 
import os
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
import importlib
import lib.tools.vector_store_manager as vsm
importlib.reload(vsm)
from lib.tools.vector_store_manager import VectorStoreManager
vs = VectorStoreManager()

response = vs.query(query_texts=["what are some action games?"], n_results=3)


print(response)

{'documents': [["[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[GameCube] Super Smash Bros. Melee (2001) - A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.']], 'distances': [[1.6755306720733643, 1.6769688129425049, 1.6769688129425049]], 'metadata': [[{'Platform': 'Nintendo 64', 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'YearOfRelease': 1996, 'Name': 'Super Mario 64', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}, {'Genre': 'Role-playing', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'Name': 'Pokémon Gold and Sil

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import importlib
import lib.tools.vector_store_manager as vsm
import lib.tools.evaluator as evalu
importlib.reload(vsm)
importlib.reload(evalu)

from lib.tools.vector_store_manager import VectorStoreManager
from lib.tools.evaluator import Evaluator
vs = VectorStoreManager()

questions=["what are some action games?", 
           "what are some multiplayer games for Nintendo platforms?", 
           "what are some racing games released in 2020?"]

for question in questions:
    response = vs.query(query_texts=[question], n_results=3)

    print(response)

    evaluator = Evaluator()
    evaluation_result = evaluator.evaluate(
        query_texts=[question],
        retrieved_docs=response
    )

    print(evaluation_result)

{'documents': [["[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[GameCube] Super Smash Bros. Melee (2001) - A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.']], 'distances': [[1.6755306720733643, 1.6769688129425049, 1.6769688129425049]], 'metadata': [[{'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Platform': 'Nintendo 64', 'Publisher': 'Nintendo', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Genre': 'Platformer'}, {'Platform': 'Game Boy Color', 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'YearOfRelease': 1999, 'Name': 'Pokémon Gold and Silver', 'Description': 'Second-g

#### Game Web Search Tool

In [ ]:
import importlib
import lib.tools.web_search as websearch
importlib.reload(websearch)

from lib.tools.web_search import WebSearchTool

tool = WebSearchTool()

tool.search("What are the most popular games in 2024?")


{
  "query": "What are the most popular games in 2024?",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://geekybrummie.com/gaming/top-50-most-notable-games-of-2024/",
      "title": "Top 50 Most Notable Games of 2024 - Geeky Brummie",
      "content": "A lot of games get released each year, and during this laidback quiet period at the end of the year, it\u2019s usually a good time to catch up on the games you might have missed. 505 Games, Rabbit and Bear Studios | PC, PlayStation, Switch, Xbox | April | Metacritic Avg: 72. SFB Games | PC, PlayStation, Switch, Xbox X/S | May | Metacritic Avg: 82. NetEase Games | PC, PS5, Xbox X/S | December | Metacritic Avg: 72. GSC Game World | PC, Xbox X/S | November | Metacritic Avg: 74. The original *Space Marine* was already a beloved game, but *Space Marine 2* feels like it\u2019s drastically ramped that up this year. Game Science | PC, PS5, Xbox X/S | August | Metacritic Avg: 79. Sony Int

### Agent

In [ ]:
import json
from typing import Optional, TypedDict

from lib.llm import LLM
from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.state_machine import EntryPoint, StateMachine, Step, Termination
from lib.tooling import tool
from lib.tools.evaluator import Evaluator
from lib.tools.vector_store_manager import VectorStoreManager
from lib.tools.web_search import WebSearchTool

vs = VectorStoreManager()
web_search_client = WebSearchTool()

def normalize_n_results(n_results, default=3):
    try:
        value = int(n_results)
    except (TypeError, ValueError):
        value = default
    return max(1, value)

@tool
def retrieve_game(query: str, n_results: int = 3):
    """Search the local vector database """
    return vs.query(query_texts=[query], n_results=normalize_n_results(n_results))

@tool
def evaluate_retrieval(question: str, n_results: int = 3):
    """Evaluate if result from vector db has answered the question"""
    safe_n_results = normalize_n_results(n_results)
    retrieved_docs = vs.query(query_texts=[question], n_results=safe_n_results)
    evaluator = Evaluator()
    return evaluator.evaluate(query_texts=[question], retrieved_docs=retrieved_docs)

@tool
def game_web_search(question: str):
    """Search the web for information."""
    return web_search_client.search(question)

class NotebookAgentState(TypedDict):
    user_query: str
    messages: list
    current_tool_calls: Optional[list]
    done: bool

agent_instructions = """
You are UdaPlay, a game research agent.
Always call retrieve_game first.
Then call evaluate_retrieval.
If evaluation says useful is false, call game_web_search.
After using tools, provide a concise final answer and mention whether it came from local retrieval or web search.
""".strip()

llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

def prepare_messages(state: NotebookAgentState):
    return {
        "messages": [
            SystemMessage(content=agent_instructions),
            UserMessage(content=state["user_query"]),
        ],
        "done": False,
    }


def call_model(state: NotebookAgentState):
    response = llm.invoke(state["messages"])
    return {
        "messages": state["messages"] + [response],
        "current_tool_calls": response.tool_calls,
        "done": not bool(response.tool_calls),
    }


def execute_tools(state: NotebookAgentState):
    tool_messages = []
    for call in state.get("current_tool_calls") or []:
        function_name = call.function.name
        function_args = json.loads(call.function.arguments)
        selected_tool = {
            "retrieve_game": retrieve_game,
            "evaluate_retrieval": evaluate_retrieval,
            "game_web_search": game_web_search,
        }[function_name]
        result = selected_tool(**function_args)
        tool_messages.append(
            ToolMessage(
                tool_call_id=call.id,
                name=function_name,
                content=json.dumps(result, default=str),
            )
        )
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None,
        "done": False,
    }


def route_after_model(state: NotebookAgentState):
    return "execute_tools" if state.get("current_tool_calls") else "__termination__"

udaplay_agent = StateMachine[NotebookAgentState](NotebookAgentState)
entry = EntryPoint[NotebookAgentState]()
prepare = Step[NotebookAgentState]("prepare_messages", prepare_messages)
model = Step[NotebookAgentState]("call_model", call_model)
tools_step = Step[NotebookAgentState]("execute_tools", execute_tools)
termination = Termination[NotebookAgentState]()

udaplay_agent.add_steps([entry, prepare, model, tools_step, termination])
udaplay_agent.connect(entry, prepare)
udaplay_agent.connect(prepare, model)
udaplay_agent.connect(model, [tools_step, termination], route_after_model)
udaplay_agent.connect(tools_step, model)

In [ ]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    run = udaplay_agent.run({
        "user_query": query,
        "messages": [],
        "current_tool_calls": None,
        "done": False,
    })
    final_state = run.get_final_state()
    final_messages = final_state["messages"]
    final_ai_messages = [message for message in final_messages if isinstance(message, AIMessage)]
    tool_messages = [message for message in final_messages if isinstance(message, ToolMessage)]
    tool_names = [message.name for message in tool_messages]

    if "game_web_search" in tool_names:
        source_label = "web_search"
    elif "retrieve_game" in tool_names:
        source_label = "local_db"
    else:
        source_label = "llm_only"

    final_answer = final_ai_messages[-1].content if final_ai_messages else "No answer generated."

    print(f"\nQuestion: {query}")
    print(f"Source: {source_label}")
    print(f"Tools Used: {', '.join(tool_names) if tool_names else 'none'}")
    print(f"Answer: {final_answer}")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare_messages
[StateMachine] Executing step: call_model
[StateMachine] Executing step: execute_tools
[StateMachine] Executing step: call_model
[StateMachine] Executing step: execute_tools
[StateMachine] Executing step: call_model
{
  "query": "When was Pok\u00e9mon Gold and Silver released?",
  "response_time": 0.65,
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.reddit.com/r/Gameboy/comments/q8rmer/pok%C3%A9mon_gold_and_silver_released_october_15_2000/",
      "title": "Pok\u00e9mon Gold and Silver released October 15, 2000",
      "content": "Pok\u00e9mon Gold and Silver released October 15, 2000 - 21 years ago these games came out and I still love them.",
      "score": 0.99979657,
      "raw_content": null
    },
    {
      "url": "https://www.facebook.com/PokemonGlobalNews/posts/today-makes-26-years-since-pok%C3%A9mon-gold-silver-were-first-release

### (Optional) Advanced

In [18]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes